[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

# Lesson 12 — Object-Oriented Programming

**Module 1 — Python Fundamentals** | ⏱ 35 min

Object-oriented programming (OOP) is a paradigm that organises code around **objects** — data structures that bundle state (attributes) and behaviour (methods) together. OOP helps manage complexity in large programs by modelling real-world entities as classes. Python is a multi-paradigm language that supports OOP naturally, with clean syntax for defining classes, inheritance, and special methods.

## Learning Objectives
- Define classes with `__init__`, instance variables, and methods
- Distinguish instance variables from class variables
- Use `@classmethod` and `@staticmethod`
- Implement inheritance and call parent methods with `super()`
- Define dunder (magic) methods: `__str__`, `__repr__`, `__len__`, `__eq__`, `__lt__`
- Use `@dataclass` to create data classes efficiently
- Implement `@property` for computed attributes and encapsulation

## Classes and Instances

A **class** is a blueprint — it defines what attributes and methods objects of that type will have. An **instance** is a specific object created from that blueprint. The `__init__` method is the **constructor** — it runs automatically when you create a new instance and sets up its initial state. The first parameter of every instance method is `self`, which refers to the specific instance being operated on.

In [ ]:
class BankAccount:
    """Represents a bank account with basic deposit/withdrawal operations."""

    # Class variable — shared by ALL instances
    interest_rate = 0.02  # 2% annual interest
    account_count = 0

    def __init__(self, owner, initial_balance=0.0):
        """Create a new bank account."""
        # Instance variables — unique to EACH instance
        self.owner = owner
        self.balance = float(initial_balance)
        self._transaction_history = []  # _ prefix = convention for 'private'

        BankAccount.account_count += 1  # Track how many accounts exist
        self.account_number = f"ACC{BankAccount.account_count:04d}"

    def deposit(self, amount):
        """Add money to the account."""
        if amount <= 0:
            raise ValueError(f"Deposit amount must be positive, got {amount}")
        self.balance += amount
        self._transaction_history.append(("deposit", amount))
        return self  # Return self to enable method chaining

    def withdraw(self, amount):
        """Remove money from the account."""
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise ValueError(f"Insufficient funds: {self.balance:.2f} < {amount:.2f}")
        self.balance -= amount
        self._transaction_history.append(("withdrawal", amount))
        return self

    def get_statement(self):
        """Return a list of all transactions."""
        return list(self._transaction_history)  # Return a copy

# Create instances
alice_account = BankAccount("Alice", 1000.00)
bob_account = BankAccount("Bob", 500.00)

print(f"{alice_account.account_number}: {alice_account.owner}, ${alice_account.balance:.2f}")
print(f"{bob_account.account_number}: {bob_account.owner}, ${bob_account.balance:.2f}")
print(f"Total accounts created: {BankAccount.account_count}")

# Method chaining — deposit().withdraw() in one line
alice_account.deposit(500).deposit(250).withdraw(100)
print(f"\nAlice's balance after transactions: ${alice_account.balance:.2f}")
print(f"Transaction history: {alice_account.get_statement()}")

## @classmethod and @staticmethod

Python methods come in three flavours. **Instance methods** (the default) receive `self` and operate on a specific instance. **Class methods** (`@classmethod`) receive `cls` (the class itself) and are used for factory methods or to access class-level data. **Static methods** (`@staticmethod`) receive neither `self` nor `cls` — they are just regular functions that logically belong to the class.

In [ ]:
class Temperature:
    """Represents a temperature value with multiple unit support."""

    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        if celsius < self.ABSOLUTE_ZERO_C:
            raise ValueError(f"Temperature below absolute zero: {celsius}°C")
        self._celsius = celsius

    # Regular instance method
    def to_fahrenheit(self):
        return self._celsius * 9/5 + 32

    def to_kelvin(self):
        return self._celsius - self.ABSOLUTE_ZERO_C

    # @classmethod — alternative constructors (factory methods)
    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        """Create a Temperature from a Fahrenheit value."""
        celsius = (fahrenheit - 32) * 5/9
        return cls(celsius)  # cls() calls __init__

    @classmethod
    def from_kelvin(cls, kelvin):
        """Create a Temperature from a Kelvin value."""
        return cls(kelvin + cls.ABSOLUTE_ZERO_C)

    # @staticmethod — utility function that belongs conceptually to the class
    @staticmethod
    def is_valid_celsius(value):
        """Check if a value is a physically valid Celsius temperature."""
        return isinstance(value, (int, float)) and value >= -273.15

    def __repr__(self):
        return f"Temperature({self._celsius:.2f}°C)"

# Using the class
room = Temperature(22.5)
print(f"{room}: {room.to_fahrenheit():.1f}°F, {room.to_kelvin():.2f}K")

body_temp = Temperature.from_fahrenheit(98.6)
print(f"Body temp: {body_temp}")

boiling = Temperature.from_kelvin(373.15)
print(f"Boiling water: {boiling}")

# Static method — called on class or instance
print(f"\n-300 valid? {Temperature.is_valid_celsius(-300)}")
print(f"20.5 valid? {Temperature.is_valid_celsius(20.5)}")

## Inheritance

Inheritance allows a class (the **subclass**) to inherit attributes and methods from another class (the **superclass** or **base class**). The subclass can add new methods, override inherited methods, or extend them using `super()`. Inheritance models "is-a" relationships — a `Dog` is an `Animal`. Python supports multiple inheritance, but use it carefully to avoid complexity.

In [ ]:
class Animal:
    """Base class for all animals."""

    def __init__(self, name, species, age):
        self.name = name
        self.species = species
        self.age = age

    def describe(self):
        return f"{self.name} is a {self.age}-year-old {self.species}"

    def speak(self):
        return "..."

    def birthday(self):
        self.age += 1
        return f"Happy birthday {self.name}! Now {self.age} years old."


class Dog(Animal):  # Dog inherits from Animal
    """Represents a dog."""

    def __init__(self, name, age, breed):
        super().__init__(name, "Canis lupus familiaris", age)  # Call parent __init__
        self.breed = breed
        self.tricks = []

    def speak(self):
        return f"{self.name} says: Woof!"

    def learn_trick(self, trick):
        self.tricks.append(trick)
        return f"{self.name} learned: {trick}"

    def describe(self):
        # Extend parent describe() rather than replace it
        base = super().describe()
        return f"{base} ({self.breed}) — tricks: {self.tricks or ['none yet']}"


class ServiceDog(Dog):  # ServiceDog inherits from Dog
    """A dog with a professional service role."""

    def __init__(self, name, age, breed, role):
        super().__init__(name, age, breed)
        self.role = role

    def describe(self):
        return f"{super().describe()} | Role: {self.role}"


# Using the hierarchy
generic = Animal("Leo", "Lion", 5)
rex = Dog("Rex", 3, "German Shepherd")
guide = ServiceDog("Buddy", 4, "Labrador", "Guide dog")

print(generic.describe())
print(rex.speak())
rex.learn_trick("sit")
rex.learn_trick("fetch")
print(rex.describe())
print(guide.describe())

# isinstance() checks inheritance
print(f"\nguide is a Dog: {isinstance(guide, Dog)}")
print(f"guide is an Animal: {isinstance(guide, Animal)}")
print(f"rex is a ServiceDog: {isinstance(rex, ServiceDog)}")

## Dunder (Magic) Methods

Dunder methods (short for "double underscore") let your classes integrate with Python's built-in operators and functions. Implementing `__str__` makes `print(obj)` show something meaningful. `__repr__` provides an unambiguous developer-facing representation. `__len__` makes `len(obj)` work. `__eq__` and `__lt__` define equality and ordering, enabling sorting and comparisons. These are what make Python's data model so elegant.

In [ ]:
from functools import total_ordering

@total_ordering  # Generates __le__, __gt__, __ge__ from __eq__ and __lt__
class Product:
    """Represents a product in an inventory system."""

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    def __str__(self):
        """User-friendly string — used by print() and str()."""
        return f"{self.name} — ${self.price:.2f} (qty: {self.quantity})"

    def __repr__(self):
        """Developer-friendly — should ideally recreate the object."""
        return f"Product({self.name!r}, {self.price}, {self.quantity})"

    def __len__(self):
        """len(product) returns the quantity."""
        return self.quantity

    def __eq__(self, other):
        """Products are equal if they have the same name and price."""
        if not isinstance(other, Product):
            return NotImplemented
        return self.name == other.name and self.price == other.price

    def __lt__(self, other):
        """Compare by price for sorting."""
        if not isinstance(other, Product):
            return NotImplemented
        return self.price < other.price

    def __add__(self, other):
        """Combine two products' quantities (must be same product)."""
        if self.name != other.name:
            raise ValueError(f"Cannot combine different products: {self.name} and {other.name}")
        return Product(self.name, self.price, self.quantity + other.quantity)


# Test dunder methods
keyboard1 = Product("Keyboard", 79.99, 50)
keyboard2 = Product("Keyboard", 79.99, 30)
mouse = Product("Mouse", 29.99, 100)
monitor = Product("Monitor", 349.00, 15)

print(str(keyboard1))     # __str__
print(repr(keyboard1))    # __repr__
print(f"Quantity: {len(keyboard1)}")  # __len__
print(f"Equal: {keyboard1 == keyboard2}")  # __eq__ — same name+price, different qty
print(f"Mouse < Keyboard: {mouse < keyboard1}")  # __lt__

# Sorting uses __lt__ (via @total_ordering)
products = [keyboard1, mouse, monitor]
print(f"\nSorted by price: {sorted(products)}")

# Addition uses __add__
combined = keyboard1 + keyboard2
print(f"\nCombined keyboards: {combined}")

## @dataclass

The `@dataclass` decorator (Python 3.7+) automatically generates boilerplate methods (`__init__`, `__repr__`, `__eq__`) based on class annotations. It dramatically reduces the code needed for simple data container classes. The `field()` function lets you customise individual fields — setting default factories, excluding fields from comparison, or marking fields as metadata.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class Employee:
    """Employee record — __init__, __repr__, __eq__ are auto-generated."""
    employee_id: int
    name: str
    department: str
    salary: float
    skills: list = field(default_factory=list)  # Correct: use field() for mutable defaults
    hire_date: datetime = field(default_factory=datetime.now)
    is_active: bool = True

    def give_raise(self, percentage):
        """Increase salary by the given percentage."""
        self.salary *= (1 + percentage / 100)
        return self

    def add_skill(self, skill):
        self.skills.append(skill)
        return self

    @property
    def annual_cost(self):
        """Total annual cost including 20% benefits overhead."""
        return self.salary * 1.20


# @dataclass generates __init__ automatically
alice = Employee(1001, "Alice Johnson", "Engineering", 95000.0)
bob = Employee(1002, "Bob Smith", "Marketing", 72000.0, is_active=True)

print(alice)  # __repr__ is auto-generated
print(bob)

alice.add_skill("Python").add_skill("SQL").add_skill("Docker")
alice.give_raise(5)  # 5% raise

print(f"\nAlice's skills: {alice.skills}")
print(f"New salary: ${alice.salary:,.2f}")
print(f"Annual cost: ${alice.annual_cost:,.2f}")

# Equality is based on all fields
alice_copy = Employee(1001, "Alice Johnson", "Engineering", 95000.0)
print(f"\nalice == alice_copy: {alice == alice_copy}")

## @property — Getters and Setters

The `@property` decorator lets you define a method that is accessed like an attribute. This is useful for **computed attributes** (values derived from other attributes) and for **validation** (checking a value before storing it). The pattern uses `@property` for the getter and `@propertyname.setter` for the setter. This enables encapsulation without changing the public interface of the class.

In [ ]:
class Circle:
    """Circle with radius validation and computed properties."""

    def __init__(self, radius):
        self.radius = radius  # This calls the setter below!

    @property
    def radius(self):
        """Get the radius."""
        return self._radius

    @radius.setter
    def radius(self, value):
        """Set the radius with validation."""
        if not isinstance(value, (int, float)):
            raise TypeError(f"Radius must be a number, got {type(value).__name__}")
        if value <= 0:
            raise ValueError(f"Radius must be positive, got {value}")
        self._radius = float(value)

    @property
    def diameter(self):
        """Computed property — no setter, diameter is read-only."""
        return self._radius * 2

    @property
    def area(self):
        import math
        return math.pi * self._radius ** 2

    @property
    def circumference(self):
        import math
        return 2 * math.pi * self._radius

    def __repr__(self):
        return f"Circle(radius={self._radius})"


c = Circle(5)
print(f"Radius:        {c.radius}")
print(f"Diameter:      {c.diameter}")
print(f"Area:          {c.area:.4f}")
print(f"Circumference: {c.circumference:.4f}")

# Setter validates the new value
c.radius = 10
print(f"\nAfter resize: {c}")
print(f"New area: {c.area:.4f}")

# Test validation
try:
    c.radius = -5  # Should raise ValueError
except ValueError as e:
    print(f"\nCaught: {e}")

try:
    c.diameter = 20  # Read-only property — raises AttributeError
except AttributeError as e:
    print(f"Caught: {e}")

## Practice Exercises

1. Create a `Stack` class with `push()`, `pop()`, `peek()`, and `is_empty()` methods. Implement `__len__`, `__repr__`, and `__contains__` (the `in` operator). Add a `max_size` parameter that raises an error when exceeded.
2. Design an inheritance hierarchy for a vehicle rental system: a base class `Vehicle` with common attributes, and subclasses `Car`, `Truck`, and `Motorcycle` each with specific attributes and an overridden `rental_cost(days)` method.
3. Convert the `Product` class from this lesson into a `@dataclass`. Add a `@property` for `total_value` (price × quantity), a validator in `__post_init__` that raises `ValueError` if price or quantity is negative, and a class method `from_dict` that creates a product from a dictionary.
4. Implement a `Vector2D` class with `@property` for `magnitude` and `angle`, dunder methods for `__add__`, `__sub__`, `__mul__` (scalar), `__str__`, and `__repr__`. Verify that `Vector2D(3, 4).magnitude == 5.0`.